In [ ]:
# ライブラリ読み込み
import os
import numpy as np
import pandas as pd
import scipy.io
from scipy.io import mmread, mmwrite
from scipy.sparse import csr_matrix
import scanpy as sc
import anndata as ann
import copy 

In [ ]:
datadir = 'MAESTER/rds_output'
matname_counts = ['A_counts_fw', 'A_counts_rev', 'C_counts_fw', 'C_counts_rev', 'G_counts_fw', 'G_counts_rev', 'T_counts_fw', 'T_counts_rev', 'coverage']

genes = pd.read_csv(os.path.join(datadir, 'genes.tsv'), sep="\t")
barcodes = np.loadtxt(os.path.join(datadir, 'barcodes.tsv'), dtype='str')
mtx = scipy.io.mmread(os.path.join(datadir, 'coverage.mtx'))
mtx = mtx.toarray()
mtx = mtx.T
adata_mg_c = ann.AnnData(X=mtx, obs=barcodes, var=genes)
for y in range(len(matname_counts)):
    mtx = scipy.io.mmread(os.path.join(datadir, matname_counts[y]+'.mtx'))
    mtx = mtx.toarray()
    mtx = mtx.T
    adata_mg_c.layers[matname_counts[y]] = mtx
adata_mg_c.var_names_make_unique() # this is unnecessary if using `var_names='gene_ids'` in `sc.read_10x_mtx`

In [ ]:
adata_mg_c

In [ ]:
adata_mg_c.obs

In [ ]:
adata_mg_c.var['refAllele']['1415']

In [ ]:
adata_mg_c.var

In [ ]:
adata_mg_c.obs = adata_mg_c.obs.rename(columns={0:'barcode'})
adata_mg_c.var = adata_mg_c.var.drop(columns='strand')

In [ ]:
adata_mg_c[0].var.reset_index()

In [ ]:
adata_mg_c[0].var.reset_index().query('refAllele!="A"').index

In [ ]:
not_A_index = np.array(adata_mg_c[0].var.reset_index().query('refAllele!="A"').index)
not_C_index = np.array(adata_mg_c[0].var.reset_index().query('refAllele!="C"').index)
not_G_index = np.array(adata_mg_c[0].var.reset_index().query('refAllele!="G"').index)
not_T_index = np.array(adata_mg_c[0].var.reset_index().query('refAllele!="T"').index)

In [ ]:
adata_mg_c_VAF = copy.deepcopy(adata_mg_c)

del adata_mg_c_VAF.layers
    
#adatas_mg_VAF.layers['variant_refA'] = adatas_mg.X - adatas_mg.layers['C_counts_fw'] - adatas_mg.layers['C_counts_rev'] - adatas_mg.layers['G_counts_fw'] - adatas_mg[x].layers['G_counts_rev'] - adatas_mg[x].layers['T_counts_fw'] - adatas_mg[x].layers['T_counts_rev']
adata_mg_c_VAF.layers['variant_refA'] = adata_mg_c.X - adata_mg_c.layers['A_counts_fw'] - adata_mg_c.layers['A_counts_rev']
adata_mg_c_VAF.layers['variant_refA'][:,not_A_index] = 0
    
#adatas_mg_VAF[x].layers['variant_refC'] = adatas_mg[x].X - adatas_mg[x].layers['A_counts_fw'] - adatas_mg[x].layers['A_counts_rev'] - adatas_mg[x].layers['G_counts_fw'] - adatas_mg[x].layers['G_counts_rev'] - adatas_mg[x].layers['T_counts_fw'] - adatas_mg[x].layers['T_counts_rev']
adata_mg_c_VAF.layers['variant_refC'] = adata_mg_c.X - adata_mg_c.layers['C_counts_fw'] - adata_mg_c.layers['C_counts_rev']
adata_mg_c_VAF.layers['variant_refC'][:,not_C_index] = 0
    
#adatas_mg_VAF[x].layers['variant_refG'] = adatas_mg[x].X - adatas_mg[x].layers['A_counts_fw'] - adatas_mg[x].layers['A_counts_rev'] - adatas_mg[x].layers['C_counts_fw'] - adatas_mg[x].layers['C_counts_rev'] - adatas_mg[x].layers['G_counts_fw'] - adatas_mg[x].layers['G_counts_rev']
adata_mg_c_VAF.layers['variant_refG'] = adata_mg_c.X - adata_mg_c.layers['G_counts_fw'] - adata_mg_c.layers['G_counts_rev']
adata_mg_c_VAF.layers['variant_refG'][:,not_G_index] = 0
    
#adatas_mg_VAF[x].layers['variant_refT'] = adatas_mg[x].X - adatas_mg[x].layers['A_counts_fw'] - adatas_mg[x].layers['A_counts_rev'] - adatas_mg[x].layers['C_counts_fw'] - adatas_mg[x].layers['C_counts_rev'] - adatas_mg[x].layers['G_counts_fw'] - adatas_mg[x].layers['G_counts_rev']
adata_mg_c_VAF.layers['variant_refT'] = adata_mg_c.X - adata_mg_c.layers['T_counts_fw'] - adata_mg_c.layers['T_counts_rev']
adata_mg_c_VAF.layers['variant_refT'][:,not_T_index] = 0
    
adata_mg_c_VAF.layers['all_mut'] = adata_mg_c_VAF.layers['variant_refA'] + adata_mg_c_VAF.layers['variant_refC'] + adata_mg_c_VAF.layers['variant_refG'] + adata_mg_c_VAF.layers['variant_refT']
adata_mg_c_VAF.layers['VAF'] = adata_mg_c_VAF.layers['all_mut'] / adata_mg_c.X

In [ ]:
adata_mg_c.layers['A_counts_rev'][:, (682)].sum()

In [ ]:
682 in not_G_index

In [ ]:
adata_mg_c.var.loc['683']

In [ ]:
layername = ['variant_refA', 'variant_refC', 'variant_refG', 'variant_refT', 'all_mut', 'VAF']

for y in range(len(layername)-2):
    del adata_mg_c_VAF.layers[layername[y]]

In [ ]:
adata_mg_c_VAF.layers['VAF'] = np.nan_to_num(adata_mg_c_VAF.layers['VAF'])

In [ ]:
#データの中に0を含む割合を表示
a = len(adata_mg_c_VAF.obs)*len(adata_mg_c_VAF.var)
print(np.count_nonzero(adata_mg_c_VAF.X == 0)/a*100)
print(np.count_nonzero(adata_mg_c_VAF.layers['all_mut'] == 0)/a*100)

In [ ]:
layername = ['all_mut', 'VAF']

for y in range(len(layername)):
    adata_mg_c_VAF.layers[layername[y]] = csr_matrix(adata_mg_c_VAF.layers[layername[y]])

In [ ]:
adata_mg_c_VAF

In [ ]:
dirname_h5ad = 'MAESTER/h5ad_files_RNA'
if not os.path.exists(dirname_h5ad):
    os.makedirs(dirname_h5ad)
    
adata_mg_c_VAF.write(os.path.join(dirname_h5ad, 'rna_mgv_counts.h5ad'), compression='gzip')

In [ ]:
# qual

In [ ]:
matname_qual = ['A_qual_fw', 'A_qual_rev', 'C_qual_fw', 'C_qual_rev', 'G_qual_fw', 'G_qual_rev', 'T_qual_fw', 'T_qual_rev', 'coverage']
genes = pd.read_csv(os.path.join(datadir, 'genes.tsv'), sep="\t")
barcodes = np.loadtxt(os.path.join(datadir, 'barcodes.tsv'), dtype='str')
mtx = scipy.io.mmread(os.path.join(datadir, 'coverage.mtx'))
mtx = mtx.toarray()
mtx = mtx.T
adata_mg_q = ann.AnnData(X=mtx, obs=barcodes, var=genes)
for y in range(len(matname_qual)):
    mtx = scipy.io.mmread(os.path.join(datadir, matname_qual[y]+'.mtx'))
    mtx = mtx.toarray()
    mtx = mtx.T
    adata_mg_q.layers[matname_qual[y]] = mtx
adata_mg_q.var_names_make_unique() # this is unnecessary if using `var_names='gene_ids'` in `sc.read_10x_mtx`

In [ ]:
adata_mg_q.obs = adata_mg_q.obs.rename(columns={0:'barcode'})
adata_mg_q.var = adata_mg_q.var.drop(columns=['strand','width'])

In [ ]:
adata_mg_q

In [ ]:
not_A_index = np.array(adata_mg_q[0].var.reset_index().query('refAllele!="A"').index)
not_C_index = np.array(adata_mg_q[0].var.reset_index().query('refAllele!="C"').index)
not_G_index = np.array(adata_mg_q[0].var.reset_index().query('refAllele!="G"').index)
not_T_index = np.array(adata_mg_q[0].var.reset_index().query('refAllele!="T"').index)

In [ ]:
adata_mg_q_VAF = copy.deepcopy(adata_mg_q)

del adata_mg_q_VAF.layers
    
#adatas_mg_VAF.layers['variant_refA'] = adatas_mg.X - adatas_mg.layers['C_counts_fw'] - adatas_mg.layers['C_counts_rev'] - adatas_mg.layers['G_counts_fw'] - adatas_mg[x].layers['G_counts_rev'] - adatas_mg[x].layers['T_counts_fw'] - adatas_mg[x].layers['T_counts_rev']
adata_mg_q_VAF.layers['variant_refA'] = adata_mg_q.X - adata_mg_q.layers['A_qual_fw'] - adata_mg_q.layers['A_qual_rev']
adata_mg_q_VAF.layers['variant_refA'][:,not_A_index] = 0
    
#adatas_mg_VAF[x].layers['variant_refC'] = adatas_mg[x].X - adatas_mg[x].layers['A_counts_fw'] - adatas_mg[x].layers['A_counts_rev'] - adatas_mg[x].layers['G_counts_fw'] - adatas_mg[x].layers['G_counts_rev'] - adatas_mg[x].layers['T_counts_fw'] - adatas_mg[x].layers['T_counts_rev']
adata_mg_q_VAF.layers['variant_refC'] = adata_mg_q.X - adata_mg_q.layers['C_qual_fw'] - adata_mg_q.layers['C_qual_rev']
adata_mg_q_VAF.layers['variant_refC'][:,not_C_index] = 0
    
#adatas_mg_VAF[x].layers['variant_refG'] = adatas_mg[x].X - adatas_mg[x].layers['A_counts_fw'] - adatas_mg[x].layers['A_counts_rev'] - adatas_mg[x].layers['C_counts_fw'] - adatas_mg[x].layers['C_counts_rev'] - adatas_mg[x].layers['G_counts_fw'] - adatas_mg[x].layers['G_counts_rev']
adata_mg_q_VAF.layers['variant_refG'] = adata_mg_q.X - adata_mg_q.layers['G_qual_fw'] - adata_mg_q.layers['G_qual_rev']
adata_mg_q_VAF.layers['variant_refG'][:,not_G_index] = 0
    
#adatas_mg_VAF[x].layers['variant_refT'] = adatas_mg[x].X - adatas_mg[x].layers['A_counts_fw'] - adatas_mg[x].layers['A_counts_rev'] - adatas_mg[x].layers['C_counts_fw'] - adatas_mg[x].layers['C_counts_rev'] - adatas_mg[x].layers['G_counts_fw'] - adatas_mg[x].layers['G_counts_rev']
adata_mg_q_VAF.layers['variant_refT'] = adata_mg_q.X - adata_mg_q.layers['T_qual_fw'] - adata_mg_q.layers['T_qual_rev']
adata_mg_q_VAF.layers['variant_refT'][:,not_T_index] = 0
    
adata_mg_q_VAF.layers['all_mut'] = adata_mg_q_VAF.layers['variant_refA'] + adata_mg_q_VAF.layers['variant_refC'] + adata_mg_q_VAF.layers['variant_refG'] + adata_mg_q_VAF.layers['variant_refT']
adata_mg_q_VAF.layers['VAF'] = adata_mg_q_VAF.layers['all_mut'] / adata_mg_q.X

In [ ]:
layername = ['variant_refA', 'variant_refC', 'variant_refG', 'variant_refT', 'all_mut', 'VAF']

for y in range(len(layername)-2):
    del adata_mg_q_VAF.layers[layername[y]]

In [ ]:
adata_mg_q_VAF.layers['VAF'] = np.nan_to_num(adata_mg_q_VAF.layers['VAF'])

In [ ]:
#データの中に0を含む割合を表示
a = len(adata_mg_q_VAF.obs)*len(adata_mg_q_VAF.var)
print(np.count_nonzero(adata_mg_q_VAF.X == 0)/a*100)
print(np.count_nonzero(adata_mg_q_VAF.layers['all_mut'] == 0)/a*100)

In [ ]:
layername = ['all_mut', 'VAF']

for y in range(len(layername)):
    adata_mg_q_VAF.layers[layername[y]] = csr_matrix(adata_mg_q_VAF.layers[layername[y]])

In [ ]:
adata_mg_q_VAF

In [ ]:
dirname_h5ad = 'MAESTER/h5ad_files_RNA'
if not os.path.exists(dirname_h5ad):
    os.makedirs(dirname_h5ad)
    
adata_mg_q_VAF.write(os.path.join(dirname_h5ad, 'rna_mgv_qual.h5ad'), compression='gzip')